# 11 - Hidden-like validation regimes (C1 / C2 / C3)

**Objective.** Assign every development query (a *training* spectrum) to the competition class it simulates, and persist that assignment once for the whole project:

* **C1** reference available: the truth is in the pool and other reference spectra of the truth remain (the query itself and its T1/T2 duplicates are excluded downstream by the frozen `test_simulated_strict` protocol);
* **C2** structure known, spectra hidden: every reference spectrum of the truth connectivity is removed, the truth stays in the pool. **Main regime.**
* **C3** structure absent: spectra removed **and** truth removed from the pool (diagnostic until a generator exists).

Regimes are assigned per truth connectivity by a seeded hash, persisted, never re-randomized. Hiding is fold-scoped. The visible `test.parquet` is never an input.

**C2 validity depends on the candidate universe** (`casmi.validation.c2_protocol`). A hidden C2 truth must look like an ordinary *external* structure -- otherwise it is the only reference-less TRAIN structure in its pool (a shortcut). The mode is decided from the universe **provenance / source counts**, never from the mere existence of `universe_manifest.json`:

| universe | `UNIVERSE_STATUS` | `C2_MODE` | C2 protocol |
|---|---|---|---|
| absent | `absent` | `PRELIMINARY_no_universe` | invalid (preliminary setup only) |
| TRAIN only | `train_only` | `PRELIMINARY_train_only_universe` | **invalid** -- Gate A fails |
| contains non-TRAIN structures (COCONUT / PubChem) | `external` | `external_universe` | valid; every C2 truth must exist in a non-TRAIN source |

**Run modes.** First run (before notebook 12): preliminary table. After notebook 12 has ingested a real external source: re-run with `REBUILD = True`. CPU only.

**Writes:** `validation/regimes/validation_regimes.parquet` (+ metadata with the signature and the universe identity), `reports/validation/`.

In [ ]:
# >>> CASMI BOOTSTRAP casmi-v2-bootstrap-2 (generated by scripts/stamp_notebooks.py; do not edit by hand)
# mount Drive -> env roots -> clone if missing, else fetch + checkout + pull --ff-only -> drop stale casmi modules -> src on sys.path
import importlib, os, subprocess, sys
from pathlib import Path
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
DRIVE_ROOT = Path(os.environ.setdefault('ENVEDA_DRIVE_ROOT', '/content/drive/MyDrive/EnvedaCASMI'))
REPO_ROOT = Path(os.environ.setdefault('ENVEDA_REPO_ROOT', '/content/Enveda'))
os.environ.setdefault('ENVEDA_SCRATCH_DIR', '/content/enveda_work')
BRANCH = os.environ.setdefault('ENVEDA_BRANCH', 'main')
SYNC_REPO = os.environ.get('ENVEDA_SYNC_REPO', '1' if IN_COLAB else '0') == '1'  # outside Colab the working tree is used as is
GITHUB_REPO = 'mybenkhadda/enveda-kaggle'


def _git(*args):
    r = subprocess.run(['git', '-C', str(REPO_ROOT), *args], capture_output=True, text=True)
    if r.returncode != 0:
        cmd = ' '.join(args)
        raise RuntimeError(f'git {cmd} failed in {REPO_ROOT}: {r.stderr.strip()} | Fix: discard local edits inside the clone '
                           f'(git -C {REPO_ROOT} stash) or delete {REPO_ROOT}, then re-run this cell; source lives on GitHub only.')
    return r.stdout.strip()


if SYNC_REPO:
    token = None
    if IN_COLAB:
        try:
            from google.colab import userdata
            token = userdata.get('GITHUB_TOKEN')  # optional Colab secret (private repository)
        except Exception:
            token = None
    url = f'https://{token}@github.com/{GITHUB_REPO}.git' if token else f'https://github.com/{GITHUB_REPO}.git'
    if not (REPO_ROOT / '.git').exists():
        r = subprocess.run(['git', 'clone', '--branch', BRANCH, url, str(REPO_ROOT)], capture_output=True, text=True)
        if r.returncode != 0:
            err = r.stderr.replace(token, '***') if token else r.stderr
            raise RuntimeError(f'git clone of {GITHUB_REPO} failed: {err.strip()} | Fix: check the GITHUB_TOKEN Colab secret / network.')
    else:
        _git('fetch', 'origin', BRANCH)
        _git('checkout', BRANCH)
        _git('pull', '--ff-only', 'origin', BRANCH)
if not (REPO_ROOT / 'src' / 'casmi' / 'workspace' / 'versions.py').exists():
    raise RuntimeError(f'{REPO_ROOT} has no src/casmi/workspace/versions.py: the source is OLDER than this notebook. '
                       f'Fix: push the bootstrap/artifact-registry refactor to GitHub {BRANCH}, restart the runtime, re-run this cell.')
for _m in [m for m in list(sys.modules) if m == 'casmi' or m.startswith('casmi.')]:
    del sys.modules[_m]  # after a pull: never mix old and new source in one kernel
importlib.invalidate_caches()
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))
os.environ['ENVEDA_BOOTSTRAP_CELL'] = 'casmi-v2-bootstrap-2'
_head = _git('log', '-1', '--format=%h %s') if (REPO_ROOT / '.git').exists() else 'no git (uploaded copy)'
print(f'source {REPO_ROOT} @ {_head} | drive {DRIVE_ROOT} | colab {IN_COLAB} | synced {SYNC_REPO}')

In [ ]:
from casmi.workspace.bootstrap import bootstrap

NOTEBOOK = '11_colab_hidden_like_validation'
REBUILD = False   # True after notebook 12 (or any universe change): rebuild + overwrite the persisted regime table
CTX = bootstrap(NOTEBOOK, notebook_api='casmi-v2-notebooks-4',
                uses_paths=('repo_root',),
                uses_artifacts=('universe_root', 'regimes_dir', 'validation_regimes', 'validation_regimes_meta', 'validation_reports_dir',
                                'regime_checks'),
                requires=('dev_queries', 'train_spectrum_metadata', 'connectivity_folds', 'molecule_mass_variants'),
                optional=('universe_manifest', 'bucket_offsets', 'universe_source_summary'),
                signatures={'casmi.validation.regimes:build_regime_table': ['queries', 'reference_connectivity', 'unified', 'cfg'],
                            'casmi.validation.regime_store:load_or_build_regimes': ['dev', 'reference_connectivity', 'unified_truths',
                                                                                    'regime_cfg', 'artifacts', 'signature', 'rebuild'],
                            'casmi.validation.c2_protocol:assess_universe': ['artifacts', 'cfg'],
                            'casmi.validation.leakage_audit:audit_regimes': ['regimes', 'train_spectrum_ids', 'fold_table',
                                                                             'reference_connectivity', 'universe_keys_sorted', 'regime_cfg',
                                                                             'c2_mode', 'rederived', 'unified_truths']})
CFG, P, ARTIFACTS, MON = CTX.cfg, CTX.paths, CTX.artifacts, CTX.monitor

## 1. Inputs

Development queries are training spectra with their truth connectivity and fold. `train_spectrum_metadata` gives the connectivity of every reference spectrum in the library -- what C1 / C2 / C3 hide or keep.

In [ ]:
import json
import numpy as np, pandas as pd

DEV_COLS = ['query_id', 'true_connectivity_key', 'fold', 'neutral_mass', 'ionization_mode', 'adduct', 'instrument_type', 'exact_mass_true']
with MON.stage('load inputs'):
    dev = pd.read_parquet(ARTIFACTS.dev_queries, columns=DEV_COLS)
    dev['query_id'] = dev['query_id'].astype(str)
    tsm = pd.read_parquet(ARTIFACTS.train_spectrum_metadata, columns=['train_spectrum_id', 'connectivity_key'])
    REF_CONN = tsm['connectivity_key'].astype(str).to_numpy()
    TRAIN_SPECTRUM_IDS = set(tsm['train_spectrum_id'].astype(str))
    folds = pd.read_parquet(ARTIFACTS.connectivity_folds)
del tsm
print(f'dev queries: {len(dev):,} | truth connectivities: {dev.true_connectivity_key.nunique():,} | reference spectra: {len(REF_CONN):,} '
      f'| fold rows: {len(folds):,} | folds: {sorted(folds.fold.unique())}')

## 2. Candidate universe status and C2 mode

`assess_universe` reads the source composition embedded in `universe_manifest.json` (or the manifests/ summary, or recounts bucket by bucket). The counts are printed explicitly because they decide whether the local C2 protocol means anything.

In [ ]:
from casmi.candidates.universe import keys_to_ids, load_candidate_keys, read_candidates
from casmi.validation.c2_protocol import C2_MODE_EXTERNAL, assess_universe

UNIVERSE = assess_universe(ARTIFACTS, CFG)
UNIVERSE_STATUS = UNIVERSE.universe_status
EXTERNAL_SOURCE_PRESENT = UNIVERSE.external_source_present
C2_PROTOCOL_VALID = UNIVERSE.c2_protocol_valid
C2_MODE = UNIVERSE.c2_mode
print('UNIVERSE_STATUS', UNIVERSE_STATUS, '| EXTERNAL_SOURCE_PRESENT', EXTERNAL_SOURCE_PRESENT, '| C2_PROTOCOL_VALID', C2_PROTOCOL_VALID,
      '| C2_MODE', C2_MODE)
print('reason:', UNIVERSE.reason)
if UNIVERSE.source_summary:
    S = UNIVERSE.source_summary
    display(pd.DataFrame([{k: S.get(k) for k in ('n_candidates', 'train_only', 'external_only', 'train_and_external', 'no_source')}]))
    display(pd.Series(S.get('per_source', {}), name='n_connectivities').to_frame())
if UNIVERSE.universe_present:
    UKEYS = load_candidate_keys(ARTIFACTS.universe_root)                 # sorted S14, memory-mapped
else:
    mv = pd.read_parquet(ARTIFACTS.molecule_mass_variants, columns=['connectivity_key'])
    UKEYS = np.sort(mv['connectivity_key'].astype(str).unique()).astype('S14')
    del mv
UNIFIED_TRUTHS = None
if C2_MODE == C2_MODE_EXTERNAL:
    tid = keys_to_ids(UKEYS, dev['true_connectivity_key'].unique())
    UNIFIED_TRUTHS = read_candidates(ARTIFACTS.universe_root, tid[tid >= 0], columns=['connectivity_key', 'candidate_sources'])
else:
    print('*** C2 IS PRELIMINARY (', C2_MODE, '): C2 truths are TRAIN structures whose references were removed; a ranker could learn')
    print('*** the candidate without references. This table is for setup only -- Gate A (notebook 13) will be protocol-invalid.')

## 3. Regime assignment

The signature records everything the assignment depends on: regime config hash, C2 mode, universe status and universe identity, number of queries. A persisted table with a different signature is never reused: the notebook stops and asks for `REBUILD = True`.

In [ ]:
from casmi.validation.regime_store import load_or_build_regimes, regime_signature
from casmi.validation.regimes import RegimeConfig, regime_counts, truth_eligibility

v = CFG['validation']
RC = RegimeConfig(split_seed=v['split_seed'], regime_fractions=v['regime_fractions'], hidden_scope=v['hidden_scope'],
                  c1_reference_protocol=v['c1_reference_protocol'], c1_min_truth_spectra=v['c1_min_truth_spectra'],
                  c2_require_external_source=bool(v['c2_require_external_source']) and C2_MODE == C2_MODE_EXTERNAL,
                  max_queries_per_regime_per_fold=v['max_queries_per_regime_per_fold'])
SIGNATURE = regime_signature(RC, C2_MODE, UNIVERSE, len(dev))
with MON.stage('regime table', n_items=len(dev), unit='queries'):
    REG, ACTION = load_or_build_regimes(dev, REF_CONN, UNIFIED_TRUTHS, RC, ARTIFACTS, SIGNATURE, rebuild=REBUILD)
print(ACTION, 'regime table | C2_MODE', C2_MODE)
print(REG['regime'].fillna('UNASSIGNED').value_counts())

In [ ]:
COUNTS = regime_counts(REG)
display(COUNTS.pivot(index='fold', columns='regime', values='n_queries').fillna(0).astype(int))
ELIG = truth_eligibility(REG['true_connectivity_key'].unique(), REF_CONN, UNIFIED_TRUTHS, RC)
ELIG_SUMMARY = {'n_truth_connectivities': int(len(ELIG)),
                'eligible_C1_rate': float(ELIG['eligible_C1'].mean()),
                'eligible_C2_rate': float(ELIG['eligible_C2'].mean()),
                'n_eligible_C2_truths': int(ELIG['eligible_C2'].sum()),
                'c2_shortcut_risk': bool(ELIG['c2_shortcut_risk'].any()),
                'c2_ineligible_reasons': ELIG.loc[~ELIG['eligible_C2'], 'c2_ineligible_reason'].value_counts().to_dict()}
print(json.dumps(ELIG_SUMMARY, indent=2))

## 4. Regime composition by strata

The regimes should differ only by what is hidden, not by population; large differences in ion mode, adduct, instrument, mass or number of spectra would confound every C1 vs C2 comparison downstream.

In [ ]:
from casmi.validation.regime_store import composition_tables

COMPOSITION, D = composition_tables(REG, dev, v['mass_bins_da'])
for col, t in COMPOSITION.items():
    print(f'--- {col} (row-normalized)')
    display(t)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
COUNTS.pivot(index='fold', columns='regime', values='n_queries').fillna(0).plot.bar(ax=axes[0], title='queries per fold / regime')
for r, g in D.groupby('regime'):
    axes[1].hist(np.log10(g['n_truth_spectra'].clip(lower=1)), bins=40, histtype='step', label=r)
    axes[2].hist(g['neutral_mass'].dropna(), bins=60, histtype='step', label=r)
axes[1].set_xlabel('log10 #spectra of the truth connectivity'); axes[1].legend()
axes[2].set_xlabel('query neutral mass (Da)'); axes[2].legend()
fig.tight_layout()
ARTIFACTS.validation_reports_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(ARTIFACTS.validation_reports_dir / 'regime_distributions.png', dpi=120)

## 5. LEAKAGE AUDIT

`casmi.validation.leakage_audit.audit_regimes` raises on any failed assertion:

1. every query is a **training** spectrum (the visible competition test set is never used for validation);
2. per fold (`assert_regime_integrity`): train / validation connectivities disjoint, one fold and one regime per connectivity, C1 truths keep >= `c1_min_truth_spectra` references, C2 and C3 truths keep **no** reference, C2 truths stay in the pool, C3 truths leave it;
3. the assignment is deterministic (rebuild == persisted table);
4. external-universe mode: every C2 truth exists in a non-TRAIN source. In a preliminary mode this is reported as a **limitation** row, never as a passed check.

The query spectrum itself and its T1/T2 duplicates are excluded from the references per query by the downstream evidence builders (notebook 14 asserts no self-matches).

In [ ]:
from casmi.validation.leakage_audit import audit_regimes
from casmi.validation.regimes import build_regime_table

with MON.stage('leakage audit'):
    REDERIVED = build_regime_table(dev, REF_CONN, unified=UNIFIED_TRUTHS, cfg=RC)
    AUDIT = audit_regimes(REG, TRAIN_SPECTRUM_IDS, folds, REF_CONN, UKEYS, RC, C2_MODE, rederived=REDERIVED, unified_truths=UNIFIED_TRUTHS)
display(AUDIT)
print('LEAKAGE AUDIT passed:', int((AUDIT['kind'] == 'assert').sum()), 'assertions |', int((AUDIT['kind'] == 'limitation').sum()),
      'documented limitation(s)')

## 6. Persist

In [ ]:
from casmi.validation.reporting import save_report
from casmi.workspace.artifacts import write_metadata

REG.to_parquet(ARTIFACTS.validation_regimes, index=False)
write_metadata(ARTIFACTS.regimes_dir, NOTEBOOK, config={'validation': CFG['validation'], 'regime_config': RC.__dict__}, seed=RC.split_seed,
               input_paths={k: ARTIFACTS.path(k) for k in ('dev_queries', 'train_spectrum_metadata', 'connectivity_folds')},
               schema_version=RC.version, repo_root=P.repo_root, filename=ARTIFACTS.validation_regimes_meta.name,
               extra={'signature': SIGNATURE, 'c2_mode': C2_MODE, 'universe': UNIVERSE.as_dict(), 'n_rows': int(len(REG)),
                      'leakage_audit': AUDIT.to_dict(orient='records'), **CTX.metadata()})
summary_table = COUNTS.assign(c2_mode=C2_MODE, regime_config_hash=RC.config_hash())
save_report(summary_table, {'notebook': NOTEBOOK, 'c2_mode': C2_MODE, 'universe_status': UNIVERSE_STATUS, 'c2_protocol_valid': C2_PROTOCOL_VALID,
                            'signature': SIGNATURE, 'eligibility': ELIG_SUMMARY, 'counts': COUNTS.to_dict(orient='records'),
                            'leakage_audit': AUDIT.to_dict(orient='records'), 'composition': {k: t.to_dict() for k, t in COMPOSITION.items()},
                            'resources': MON.table().to_dict(orient='records'),
                            'note': 'regime definitions + counts only; no model metric is computed in this notebook'},
            ARTIFACTS.validation_reports_dir, 'validation_summary')
AUDIT.to_parquet(ARTIFACTS.regime_checks, index=False)
print('saved', ARTIFACTS.validation_regimes)
if not UNIVERSE.universe_present:
    print('NEXT: 12_colab_candidate_universe')
elif not C2_PROTOCOL_VALID:
    print('NEXT: provide an external source (COCONUT at data/external/coconut/coconut.csv) -> 12 -> 11 with REBUILD = True')
else:
    print('NEXT: 13_colab_candidate_recall (Gate A)')